## Initialisations

In [3]:
# Checking Python executable
import sys
print(sys.executable) # MAKE SURE THIS POINTS TO THE CORRECT VIRTUAL ENVIRONMENT PATH FOR CORRECT PACKAGE INSTALLATION

/home/louis/miniconda3/envs/aml_lab/bin/python


In [4]:
# ALWAYS INSTALL USING %pip, NOT !pip (can sometimes install to system Python) or pip
# %pip install numpy
# %pip install pandas
# %pip install matplotlib
# %pip install scikit-learn
# %pip install torch # Using version 2.10.0+cu128
# %pip install torchinfo

## Setup

### Imports and Training Definitions

In [5]:
# Import packages
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchinfo
from preprocessing import create_training_dataset, LABEL_MAP, FEATURE_COLS
from custom_loss import CostSensitiveLoss, penalty_grid
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import torch.nn.functional as F
print(torch.__version__)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # Define device (either GPU or CPU if GPU is unavailable)
# DEVICE = "cpu"

##### Model training function #####
def train(
        model: nn.Module,
        train_loader: DataLoader,
        criterion: nn.Module,
        optimizer: torch.optim.Optimizer,
        num_epochs: int = 10,
        val_loader: DataLoader = None,
        device: torch.device = DEVICE,
        print_loss: bool = True, # Flag for whether to print loss outputs or not
):
    model = model.to(device) # Move the model to same device as data (GPU or CPU)

    least_val_loss = 100 # Initialise best validation accuracy
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

    # Train for the number of epochs specified
    for epoch in range(num_epochs):

        ### TRAINING SET ###
        model.train() # Set model to training mode (affects Dropout/BatchNorm)
        train_loss = 0.0 # Initialise training loss

        # Loop through all batches in the training set
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device) # Move data to same device as model (GPU or CPU)
            optimizer.zero_grad() # Clear gradients
            preds = model(inputs) # Forward pass, obtain predictions
            loss = criterion(preds, labels) # Compute loss based on predictions and true labels (loss is MEAN loss over the batch)
            loss.backward() # Backward pass, compute gradient of loss w.r.t every model parameter
            optimizer.step() # Update weights, using optimisation algorithm chosen

            train_loss += loss.item() * inputs.size(0) # Sum training loss of EACH SAMPLE in the batch (inputs.size(0) is batch size)
        train_loss /= len(train_loader.dataset) # Calculate mean loss PER SAMPLE over ENTIRE DATASET

        ### VALIDATION SET ###
        # Loop through all validation batches (if validation data is given)
        if val_loader is not None:
            model.eval() # Set model to evaluation (inference) mode (turns dropout OFF, and affects BatchNorm)
            val_loss = 0.0 # Initialise validation loss
            all_preds_class = [] # Initialise list to store output prediction classes
            all_labels = [] # Initialise list to store actual labels of output predictions

            with torch.no_grad(): # Disable gradient computing
                # Loop through all batches in the validation set
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device) # Move data to same device as model (GPU or CPU)
                    preds = model(inputs) # Forward pass, obtain predictions as LOGITS (NO FOLLOWING BACKWARD PASS IN VALIDATION)

                    # Compute confusion matrix values
                    preds_class = torch.argmax(preds, dim=1) # Get class index of logit predictions
                    all_preds_class.append(preds_class.cpu())
                    all_labels.append(labels.cpu())

                    # Compute validation loss
                    loss = criterion(preds, labels) # Compute loss based on predictions and true labels (loss is MEAN loss over the batch)
                    val_loss += loss.item() * inputs.size(0) # Sum validation loss of EACH SAMPLE in the batch (inputs.size(0) is batch size)
                val_loss /= len(val_loader.dataset)

                all_preds_class = torch.cat(all_preds_class)
                all_labels = torch.cat(all_labels)

                cm = confusion_matrix(all_labels, all_preds_class)
                print("Validation confusion matrix:\n", cm)

                scheduler.step(val_loss)

                if val_loss < least_val_loss:
                    print("FOUND BEST")
                    least_val_loss = val_loss
                    torch.save(model.state_dict(), model_save_dir)

        ### PRINT TRAINING/VALIDATION OUTPUTS ###
            if print_loss:
                print(f"Epoch[{epoch+1}/{num_epochs}] Training Loss: {train_loss:.5f}, Validation Loss: {val_loss:.5f}")
        else:
            if print_loss:
                print(f"Epoch[{epoch+1}/{num_epochs}] Training Loss: {train_loss:.5f}")


##### Model evaluation function #####
def eval(
        model: nn.Module,
        test_loader: DataLoader,
        criterion: nn.Module,
        device: torch.device = DEVICE,
        print_loss: bool = True, # Flag for whether to print loss outputs or not
):
    model.eval() # Set model to evaluation mode
    test_loss = 0.0 # Initialise test loss
    all_preds_class = [] # Initialise list to store output prediction classes
    all_labels = [] # Initialise list to store actual labels of output predictions

    with torch.no_grad(): # Disable gradient computing
        # Loop through all batches in the test set
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device) # Move data to same device as model (GPU or CPU)
            preds = model(inputs) # Forward pass, obtain predictions as LOGITS (NO FOLLOWING BACKWARD PASS IN TESTING)

            # Compute confusion matrix values
            preds_class = torch.argmax(preds, dim=1) # Get class index of logit predictions
            all_preds_class.append(preds_class.cpu())
            all_labels.append(labels.cpu())
            
            # Compute validation loss
            loss = criterion(preds, labels) # Compute loss based on predictions and true labels (loss is MEAN loss over the batch)

            test_loss += loss.item() * inputs.size(0) # Sum test loss of EACH SAMPLE in the batch (inputs.size(0) is batch size)
        test_loss /= len(test_loader.dataset)

        all_preds_class = torch.cat(all_preds_class)
        all_labels = torch.cat(all_labels)

        cm = confusion_matrix(all_labels, all_preds_class)
        print("Testing confusion matrix:\n", cm)

    if print_loss:
        print(f"Test Loss: {test_loss:.5f}")

2.10.0+cu128


### Setting Constants

In [6]:
### Define constants and variables ###
num_sensor_readings = 14 # Number of sensor readings
num_classes = 4 # Number of classification classes (number of emotional states to identify)
BATCH_SIZE = 64 # Batch size

WINDOW_SIZE = 125
STEP_SIZE = 32

model_save_dir = "Ml-Models/best_emotion_vgg_lstm2.pth"

## Data Loading and Processing

### Import and Create Datasets

In [7]:
### Import data and create training datasets and labels ###
data_dir = "EmoRecData/" # Define dataset directory

# ignore_files=["EmoRecData/emm4_7_5min_baseline.csv",'EmoRecData/emm4_7_5min_distract.csv',"EmoRecData/emm4_7_5min_stress.csv",'EmoRecData/emm4_7_5min_focus.csv']

# Create training datasets and labels
# X, y_int, label_reg=create_training_dataset(data_dir,None) # Outputs: Input channels, Labels, One hot labels
X, y_int, label_encoder=create_training_dataset(data_dir,None) # Outputs: Input channels, Labels, One hot labels
label_encoder.fit(["distracted", "focused", "relaxed", "stressed"])

# DEBUGGING
print(X.shape)

Skipping test file: adi_7_5min_baseline.csv
Skipping test file: adi_7_5min_distract.csv
Skipping test file: adi_7_5min_focus.csv
Skipping test file: adi_7_5min_stress.csv
Skipping test file: adi_focused.csv
Skipping test file: emm4_7_5min_baseline.csv
Skipping test file: emm4_7_5min_distract.csv
Skipping test file: emm4_7_5min_focus.csv
Skipping test file: emm4_7_5min_stress.csv
Skipping test file: emmanuel_7_5min_baseline.csv
Skipping test file: emmanuel_7_5min_distract.csv
Skipping test file: emmanuel_7_5min_focus.csv
Skipping test file: emmanuel_7_5min_stress.csv
Skipping test file: focus_test.csv
Skipping test file: louis_7_5min_baseline.csv
Skipping test file: louis_7_5min_distract.csv
Skipping test file: louis_7_5min_focus.csv
Skipping test file: louis_7_5min_stress.csv
Skipping test file: louis_focused.csv
Skipping test file: louis_stressed.csv
Using 40 training files (ignored 20 test/backup files)
Processing adi1_7_5min_baseline.csv...
Processing adi1_7_5min_distract.csv...
Pro

### Split Datasets

In [8]:
### Split data into training and validation sets ###
# label_encoder = LabelEncoder()

# Train val split
X_train, X_val, y_train, y_val = train_test_split(
    X, y_int, test_size=0.2, random_state=42, stratify=y_int
)

### Form dataloaders ###
X_train_torch = torch.FloatTensor(X_train).transpose(1, 2) 
X_val_torch   = torch.FloatTensor(X_val).transpose(1, 2)
y_train_torch = torch.LongTensor(y_train)
y_val_torch   = torch.LongTensor(y_val)

train_dataset = TensorDataset(X_train_torch, y_train_torch)
val_dataset   = TensorDataset(X_val_torch,   y_val_torch)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

for batch_X, batch_y in train_loader:
    print(f"FIRST BATCH shape: {batch_X.shape}")
    break



# DEBUGGING
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print(f"X_train_torch shape: {X_train_torch.shape}")
print(f"X_val_torch shape:   {X_val_torch.shape}")

print(X_train.shape)


FIRST BATCH shape: torch.Size([64, 14, 125])
Using device: cuda
X_train_torch shape: torch.Size([14430, 14, 125])
X_val_torch shape:   torch.Size([3608, 14, 125])
(14430, 125, 14)


## Model Definitions

### LSTM

In [7]:
##### Model definition #####
class LSTMClassifier(nn.Module):
    def __init__(self, input_size=num_sensor_readings, hidden_size=64, output_size=num_classes): # Note: output_size should be equal to number of classification classes
        super().__init__()
        self.lstm1 = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True, dropout=0.3) # Input: [batch, sequence length, input dimension (number of sensors)]
        # self.relu1 = nn.ReLU() # ReLU
        self.dropout1 = nn.Dropout(0.2) # Dropout

        self.lstm2 = nn.LSTM(input_size=hidden_size, hidden_size=hidden_size, batch_first=True, dropout=0.3) # Input: [batch, sequence length, input dimension (number of sensors)]
        # self.relu2 = nn.ReLU() # ReLU
        self.dropout2 = nn.Dropout(0.2) # Dropout

        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.layernorm1 = nn.LayerNorm(hidden_size) # Layer norm
        self.relu3 = nn.ReLU() # ReLU
        self.dropout3 = nn.Dropout(0.2) # Dropout

        self.fc2 = nn.Linear(hidden_size, output_size)
        # self.layernorm2 = nn.LayerNorm(output_size) # Layer norm
        # self.relu4 = nn.ReLU() # ReLU
        # self.dropout4 = nn.Dropout(0.3) # Dropout
    
    def forward(self, x):
        x = x.permute(0,2,1)

        out, _ = self.lstm1(x) # Output: [batch, sequence length, hidden dimension (number of sensors)]
        # out = out[:, -1, :] # Use final hidden state of model as the output classification (Output: [batch, hidden dimension])
        # out = self.relu1(out)
        out = self.dropout1(out)
        
        out, _ = self.lstm2(out)
        # out = self.relu2(out)
        out = self.dropout2(out)
        out = out[:, -1, :]
        
        out = self.fc1(out) # CLASSIFY: Output here are LOGITS (Output: [batch, num_classes])
        out = self.layernorm1(out)
        out = self.relu3(out)
        out = self.dropout3(out)

        out = self.fc2(out) # CLASSIFY: Output here are LOGITS (Output: [batch, num_classes])
        # out = self.layernorm2(out)
        # out = self.relu4(out)
        # out_logits = self.dropout4(out)

        return out # AS LOGITS

### 1D CNN

In [ ]:
# Train REGULARIZED version
class EmotionCNN_Reg(nn.Module):
    def __init__(self, input_channels=14, num_classes=4):
        super().__init__()
        self.conv1 = nn.Conv1d(input_channels, 16, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm1d(16)
        self.pool1 = nn.MaxPool1d(2)

        self.conv2 = nn.Conv1d(16, 32, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm1d(32)
        self.pool2 = nn.MaxPool1d(2)

        #self.fc1       = nn.Linear(32 * 4, 64) # use for window size =16 / step size 4
        self.fc1       = nn.Linear(32 * 31, 64)
        self.dropout1  = nn.Dropout(0.3)
        self.fc2       = nn.Linear(64, 32)
        self.dropout2  = nn.Dropout(0.5)
        self.fc3       = nn.Linear(32, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool1(x)
        x = F.dropout(x, 0.2)

        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool2(x)

        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        x = self.fc3(x)
        return x

# Retrain with regularization
model_reg = EmotionCNN_Reg(input_channels=X_train_torch.shape[1],
                           num_classes=4).to(device)
optimizer_reg = optim.Adam(model_reg.parameters(), lr=0.001, weight_decay=1e-3)
criterion = nn.CrossEntropyLoss()

print("🛡️ Training regularized model...")
print("Input channels:", X_train_torch.shape[1])
print("Total params:", sum(p.numel() for p in model_reg.parameters()))


### VGG

In [15]:
# VGG
# Create a convolutional neural network 
class EmotionCNN_RegV2(nn.Module):
    """
    Model architecture copying TinyVGG from: 
    https://poloclub.github.io/cnn-explainer/
    """
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
        super().__init__()
        self.block_1 = nn.Sequential(
            nn.Conv1d(in_channels=input_shape, 
                      out_channels=hidden_units, 
                      kernel_size=3, 
                      stride=1, 
                      padding=1), 
            nn.BatchNorm1d(hidden_units),
            nn.ReLU(),
            nn.Conv1d(in_channels=hidden_units, 
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2,
                         stride=2) # default stride value is same as kernel_size
        )
        self.block_2 = nn.Sequential(
            nn.Conv1d(hidden_units, hidden_units, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_units),
            nn.Conv1d(hidden_units, hidden_units, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            # Where did this in_features shape come from? 
            # It's because each layer of our network compresses and changes the shape of our input data.
            nn.Linear(in_features=hidden_units*31 ,out_features=output_shape)
        )
    
    def forward(self, x: torch.Tensor):
        x = self.block_1(x)
        # print(x.shape)
        x = self.block_2(x)
        # print(x.shape)
        x = self.classifier(x)
        # print(x.shape)
        return x

torch.manual_seed(42)
model_2 = EmotionCNN_RegV2(input_shape=14,hidden_units=10, output_shape=4).to(device)
# model_2

### CNN + LSTM

In [67]:
##### Model definition #####
class CNN_LSTMClassifier(nn.Module):
    def __init__(self, input_size=num_sensor_readings, hidden_size=64, output_size=num_classes): # Note: output_size should be equal to number of classification classes
        super().__init__()

        ### CNN Feature Classifier ###
        self.conv1 = nn.Conv1d(input_size, 64, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(2)

        self.conv2 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(128)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool1d(2)

        ### LSTM ###
        self.lstm1 = nn.LSTM(input_size=128, hidden_size=hidden_size, batch_first=True, dropout=0.3) # Input: [batch, sequence length, input dimension (number of sensors)]
        self.dropout1 = nn.Dropout(0.2) # Dropout

        self.lstm2 = nn.LSTM(input_size=hidden_size, hidden_size=hidden_size, batch_first=True, dropout=0.3) # Input: [batch, sequence length, input dimension (number of sensors)]
        self.dropout2 = nn.Dropout(0.2) # Dropout

        ### Classifier ###
        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.layernorm1 = nn.LayerNorm(hidden_size) # Layer norm
        self.relu3 = nn.ReLU() # ReLU
        self.dropout3 = nn.Dropout(0.2) # Dropout

        self.fc2 = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        ### CNN ###
        # x = self.conv1(x)
        # x = self.bn1(x)
        # x = self.relu1(x)
        # x = self.pool1(x)
        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))

        # x = self.conv2(x)
        # x = self.bn2(x)
        # x = self.relu2(x)
        # x = self.pool2(x)
        x = self.pool2(self.relu2(self.bn2(self.conv2(x))))

        ### LSTM ###
        x = x.permute(0,2,1) # Convert to LSTM format

        out, _ = self.lstm1(x) # Output: [batch, sequence length, hidden dimension (number of sensors)]
        out = self.dropout1(out)
        
        out, _ = self.lstm2(out)
        out = self.dropout2(out)

        out = out[:, -1, :] # Use final hidden state of model as the output classification (Output: [batch, hidden dimension])
        
        ### Classifier ###
        out = self.fc1(out) # CLASSIFY: Output here are LOGITS (Output: [batch, num_classes])
        out = self.layernorm1(out)
        out = self.relu3(out)
        out = self.dropout3(out)

        out = self.fc2(out) # CLASSIFY: Output here are LOGITS (Output: [batch, num_classes])

        return out # AS LOGITS

### Simple CNN + LSTM

In [11]:
##### Model definition #####
class simple_CNN_LSTMClassifier(nn.Module):
    def __init__(self, input_size=num_sensor_readings, hidden_size=64, output_size=num_classes): # Note: output_size should be equal to number of classification classes
        super().__init__()

        ### CNN Feature Classifier ###
        self.conv1 = nn.Conv1d(input_size, 10, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(10)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(2)

        self.conv2 = nn.Conv1d(10, 10, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(10)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool1d(2)

        ### LSTM ###
        self.lstm1 = nn.LSTM(input_size=10, hidden_size=5, batch_first=True) # Input: [batch, sequence length, input dimension (number of sensors)]

        ### Classifier ###
        self.fc1 = nn.Linear(5, output_size)
    
    def forward(self, x):
        ### CNN ###
        # x = self.conv1(x)
        # x = self.relu1(x)
        # x = self.pool1(x)
        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu2(self.bn2(self.conv2(x))))

        ### LSTM ###
        x = x.permute(0,2,1) # Convert to LSTM format

        out, _ = self.lstm1(x) # Output: [batch, sequence length, hidden dimension (number of sensors)]

        out = out[:, -1, :] # Use final hidden state of model as the output classification (Output: [batch, hidden dimension])
        
        ### Classifier ###
        out = self.fc1(out) # CLASSIFY: Output here are LOGITS (Output: [batch, num_classes])

        return out # AS LOGITS

### VGG + LSTM

In [9]:
# VGG
# Create a convolutional neural network 
class VGG_LSTM(nn.Module):
    """
    Model architecture copying TinyVGG from: 
    https://poloclub.github.io/cnn-explainer/
    """
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
        super().__init__()
        self.block_1 = nn.Sequential(
            nn.Conv1d(in_channels=input_shape, 
                      out_channels=hidden_units, 
                      kernel_size=3, 
                      stride=1, 
                      padding=1), 
            nn.BatchNorm1d(hidden_units),
            nn.ReLU(),
            nn.Conv1d(in_channels=hidden_units, 
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2,
                         stride=2) # default stride value is same as kernel_size
        )
        self.block_2 = nn.Sequential(
            nn.Conv1d(hidden_units, hidden_units, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_units),
            nn.Conv1d(hidden_units, hidden_units, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )
        self.block_3 = nn.Sequential(
            nn.LSTM(input_size=31, hidden_size=10, batch_first=True) # Input: [batch, sequence length, input dimension (number of sensors)]
        )
        self.classifier = nn.Sequential(
            # nn.Flatten(),
            # Where did this in_features shape come from? 
            # It's because each layer of our network compresses and changes the shape of our input data.
            nn.Linear(in_features=10 ,out_features=output_shape)
        )
    
    def forward(self, x: torch.Tensor):
        x = self.block_1(x)
        # print(x.shape)
        x = self.block_2(x)
        # print(x.shape)
        x, _ = self.block_3(x)
        x = x[:, -1, :]
        x = self.classifier(x)
        # print(x.shape)
        return x

torch.manual_seed(42)
model = VGG_LSTM(input_shape=14,hidden_units=10, output_shape=4).to(device)

print(torchinfo.summary(model, input_size=(1, num_sensor_readings, WINDOW_SIZE))) # Input: [batch size, sequence length, input size (number of sensors)]

Layer (type:depth-idx)                   Output Shape              Param #
VGG_LSTM                                 [1, 4]                    --
├─Sequential: 1-1                        [1, 10, 62]               --
│    └─Conv1d: 2-1                       [1, 10, 125]              430
│    └─BatchNorm1d: 2-2                  [1, 10, 125]              20
│    └─ReLU: 2-3                         [1, 10, 125]              --
│    └─Conv1d: 2-4                       [1, 10, 125]              310
│    └─ReLU: 2-5                         [1, 10, 125]              --
│    └─MaxPool1d: 2-6                    [1, 10, 62]               --
├─Sequential: 1-2                        [1, 10, 31]               --
│    └─Conv1d: 2-7                       [1, 10, 62]               310
│    └─ReLU: 2-8                         [1, 10, 62]               --
│    └─BatchNorm1d: 2-9                  [1, 10, 62]               20
│    └─Conv1d: 2-10                      [1, 10, 62]               310
│    └─ReLU

## Model Training and Evaluation

### Train Model

In [10]:
##### Traing model #####
# Define model
# model = LSTMClassifier()#.to(DEVICE) # LSTM
# model = CNN_LSTMClassifier().to(DEVICE) # CNN + LSTM
# model = simple_CNN_LSTMClassifier().to(DEVICE) # Simpler CNN + LSTM
model = VGG_LSTM(input_shape=14,hidden_units=10, output_shape=4).to(DEVICE) # VGG + LSTM
print(torchinfo.summary(model, input_size=(1, num_sensor_readings, WINDOW_SIZE))) # Input: [batch size, sequence length, input size (number of sensors)]
customLoss = CostSensitiveLoss(cost_matrix=penalty_grid)

# Train model
train(
    model,
    train_loader,
    # nn.CrossEntropyLoss(), #nn.CrossEntropyLoss() for multi-class classification, nn.BCEWithLogitsLoss() for binary classification, nn.MSELoss()
    customLoss,
    # optim.Adam(model.parameters(), lr=0.001),
    optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-3),
    # optim.SGD(model.parameters(), lr=0.01, momentum=0.9),
    num_epochs=150,#500
    val_loader=val_loader,
    print_loss=True,
)


# # Test model
# eval(
#     model,
#     test_loader,
#     nn.CrossEntropyLoss(), #nn.CrossEntropyLoss() for multi-class classification, nn.BCEWithLogitsLoss() for binary classification, nn.MSELoss()
#     print_loss=True,
# )

Layer (type:depth-idx)                   Output Shape              Param #
VGG_LSTM                                 [1, 4]                    --
├─Sequential: 1-1                        [1, 10, 62]               --
│    └─Conv1d: 2-1                       [1, 10, 125]              430
│    └─BatchNorm1d: 2-2                  [1, 10, 125]              20
│    └─ReLU: 2-3                         [1, 10, 125]              --
│    └─Conv1d: 2-4                       [1, 10, 125]              310
│    └─ReLU: 2-5                         [1, 10, 125]              --
│    └─MaxPool1d: 2-6                    [1, 10, 62]               --
├─Sequential: 1-2                        [1, 10, 31]               --
│    └─Conv1d: 2-7                       [1, 10, 62]               310
│    └─ReLU: 2-8                         [1, 10, 62]               --
│    └─BatchNorm1d: 2-9                  [1, 10, 62]               20
│    └─Conv1d: 2-10                      [1, 10, 62]               310
│    └─ReLU

### Evaluate Model

In [17]:
##### Evaluation on test data #####
### Function for inferring label from filename ###
def infer_label_from_filename(fname: str) -> str:
    lower = fname.lower()
    for key, lab in LABEL_MAP.items():
        if key in lower:
            return lab
    raise ValueError(f"Could not infer label for: {fname}")

### Function for getting final emotion prediction/label from a file ###
def predict_file_reg(fpath: str, model_reg, label_encoder):
    """Predict emotions on new file using regularized model."""
    df = pd.read_csv(fpath)

    # Convert objects to numeric (same as training)
    for col in FEATURE_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    data = df[FEATURE_COLS].dropna().values 

    windows = []
    start = 0
    while start + WINDOW_SIZE <= len(data):
        window = data[start:start + WINDOW_SIZE] 

        window_mean = np.mean(window, axis=0, keepdims=True)
        window_std  = np.std(window, axis=0, keepdims=True)
        window_norm = (window - window_mean) / (window_std + 1e-8)

        windows.append(window_norm)
        start += STEP_SIZE

    if not windows:
        return None, None

    X_new = np.stack(windows)  # (n_windows, 125, 14)
    X_new_torch = torch.FloatTensor(X_new).transpose(1, 2).to(device)  

    model_reg.eval()
    with torch.no_grad():
        outputs = model_reg(X_new_torch)
        probs = F.softmax(outputs, dim=1).cpu().numpy()
        preds = np.argmax(probs, axis=1)

    labels = label_encoder.inverse_transform(preds)
    return labels, probs


### Define test file ###
# test_file = "/content/drive/MyDrive/EmoRecData/louis_stressed.csv" # change path
# test_file = "EmoRecData/louis_7_5min_focus.csv" # change path
# test_file = "EmoRecData/adi_7_5min_focus.csv" # change path
# test_file = "EmoRecData/emmanuel_7_5min_focus.csv" # change path
# test_file = "EmoRecData/emm4_7_5min_focus.csv" # change path

# test_file = "EmoRecData/emmanuel_7_5min_baseline.csv" # change path
# test_file = "EmoRecData/adi_7_5min_baseline.csv" # change path
# test_file = "EmoRecData/louis_7_5min_baseline.csv" # change path
# test_file = "EmoRecData/emm4_7_5min_baseline.csv" # change path

# test_file = "EmoRecData/louis_7_5min_stress.csv" # change path
# test_file = "EmoRecData/adi_7_5min_stress.csv" # change path
# test_file = "EmoRecData/emmanuel_7_5min_stress.csv" # change path
# test_file = "EmoRecData/emm4_7_5min_stress.csv" # change path

# test_file = "EmoRecData/louis_7_5min_distract.csv" # change path
# test_file = "EmoRecData/adi_7_5min_distract.csv" # change path
test_file = "EmoRecData/emm4_7_5min_distract.csv" # change path


# label_encoder = LabelEncoder()
# label_encoder.fit(["relaxed", "focused", "distracted", "stressed"])

# labels_reg, probs_reg = predict_file_reg(test_file, model_reg, label_encoder)

# Call best model weights
# 1D CNN
# model = model_reg
# model.load_state_dict(torch.load("Ml-Models/cnn_without_orientation_data.pth"))

# LSTM
# model = LSTMClassifier().to(DEVICE)
# model.load_state_dict(torch.load("Ml-Models/best_emotion_lstm_customLoss.pth"))
# model.load_state_dict(torch.load("Ml-Models/best_emotion_lstm_noNorm1.pth"))

# VGG
# model = model_2
# model.load_state_dict(torch.load("emotionv2_tiny_vgg_updated_windowing_14features.pth"))

# CNN + LSTM
# model = simple_CNN_LSTMClassifier().to(DEVICE)
# # model.load_state_dict(torch.load("Ml-Models/best_emotion_cnn_lstm1.pth"))
# model.load_state_dict(torch.load(model_save_dir))

# VGG + LSTM
model = VGG_LSTM(input_shape=14,hidden_units=10, output_shape=4).to(DEVICE)
model.load_state_dict(torch.load(model_save_dir))

labels_reg, probs_reg = predict_file_reg(test_file, model, label_encoder)

filename = os.path.basename(test_file)
print(f"🛡️ REGULARIZED MODEL (92% val acc) on {filename}:")
print(f"🎯 {len(labels_reg)} windows predicted")
print(f"Most common counts: {np.bincount([label_encoder.transform([l])[0] for l in labels_reg], minlength=4)}")

print("\nFirst 10 predictions:")
for i, pred in enumerate(labels_reg[:10]):
# for i, pred in enumerate(labels_reg[:346]):
    print(f"  {i+1:2d}: {pred}")

print("\nPREDICTION DISTRIBUTION:")
unique, counts = np.unique(labels_reg, return_counts=True)
total_windows = len(labels_reg)
for label, count in zip(unique, counts):
    pct = count / total_windows * 100
    print(f"  {label:10s}: {count}/{total_windows} ({pct:.0f}%)")

# Top prediction
top_class_idx = np.argmax(np.bincount([label_encoder.transform([l])[0] for l in labels_reg]))
top_class = label_encoder.classes_[top_class_idx]
top_pct = 100 * np.max(np.bincount([label_encoder.transform([l])[0] for l in labels_reg])) / len(labels_reg)
true_label = infer_label_from_filename(filename)
print(f"\n TOP PREDICTION: {top_class} ({top_pct:.0f}%)")
print(f" TRUE LABEL:     {true_label}")

🛡️ REGULARIZED MODEL (92% val acc) on emm4_7_5min_distract.csv:
🎯 412 windows predicted
Most common counts: [  0 411   1   0]

First 10 predictions:
   1: focused
   2: focused
   3: focused
   4: focused
   5: focused
   6: focused
   7: focused
   8: focused
   9: focused
  10: focused

PREDICTION DISTRIBUTION:
  focused   : 411/412 (100%)
  relaxed   : 1/412 (0%)

 TOP PREDICTION: focused (100%)
 TRUE LABEL:     distracted
